*Figure notebook of our analysis (IDIBAPS cFos brains). It reads the project settings from the environment variable `CFOS_MAPPER_SETTINGS` (or `./settings.json`); set it before running:* `%env CFOS_MAPPER_SETTINGS=/path/to/settings.json`

# Step 5 - figures of the results (coronal view)

Kernel: **cellfinder-latest (1.11)**

Uses the whole-brain results of `4_run_brain.py` (`results/<brain>/full_brain/<run_name>/`) for every brain that has them, and makes:
1. cells per brain (total and per mm³)
2. coronal view of cell density (all cells, projected from front to back)
3. coronal slabs of cell density from front to back
4. cell density along the anterior–posterior axis
5. a coronal close-up of the cFos image with the detected cells
6. a short video flying from front to back through that close-up (mp4)
7. a 3D view of one brain: raw cFos, cell density and cells
8. a video flying from front to back through the whole brain: coronal section (2D) next to the 3D brain (mp4)
9. a video of the 3D brain turning around (mp4)

Axes of the data: z = dorsal (plane 0) → ventral, y = anterior → posterior, x = left–right. Coronal = z (up–down) × x (left–right), dorsal up.

Figures are saved as PDF (editable in Illustrator) and PNG (slides) in `figures/results_<run_name>/`.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cfos_mapper as cf
from cfos_mapper.figures import style as fs

fs.apply_style()
settings = cf.load_settings()
results = Path(settings['results_folder'])

# the network used in 4_run_brain.py (= folder name in results/<brain>/full_brain/)
run_name = 'network_v2'

brains = sorted(p.parents[2].name for p in results.glob(f'*/full_brain/{run_name}/cells.csv'))
out = results.parent / 'figures' / f'results_{run_name}'
print('brains with results:', brains)

## Load the cells and the outline of each brain

In [ ]:
data = {}
for brain in brains:
    run = results / brain / 'full_brain' / run_name
    info = json.load(open(run / 'run_info.json'))
    cells = pd.read_csv(run / 'cells.csv')
    voxel_um = np.array(info['voxel_size'])

    # brain outline at low resolution, only for the planes that were analysed
    mask, mask_um = cf.tissue_mask(brain, settings)
    mask_um = np.array(mask_um)
    z0, z1 = [int(round(p * voxel_um[0] / mask_um[0])) for p in info['planes']]
    mask[:z0] = False
    mask[z1:] = False

    # cell positions in micrometers
    um = pd.DataFrame({axis: cells[axis] * voxel_um[i] for i, axis in enumerate('zyx')})
    data[brain] = dict(cells=cells, um=um, info=info, voxel_um=voxel_um, mask=mask, mask_um=mask_um,
                       volume_mm3=mask.sum() * np.prod(mask_um) / 1e9)

summary = pd.DataFrame([{
    'brain': brain,
    'cells': len(d['cells']),
    'brain volume (mm³)': round(d['volume_mm3'], 1),
    'cells per mm³': round(len(d['cells']) / d['volume_mm3']),
    'candidates': d['info']['candidates'],
    'planes': '{}-{}'.format(*d['info']['planes']),
} for brain, d in data.items()])
out.mkdir(parents=True, exist_ok=True)
summary.to_csv(out / 'summary.csv', index=False)
summary

## 1. Cells per brain

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(4 + 1.6 * len(brains), 4.2))
for ax, column, title in [(axes[0], 'cells', 'cFos+ cells per brain'),
                          (axes[1], 'cells per mm³', 'Cell density')]:
    bars = ax.bar(summary.brain, summary[column], color=fs.BLUE, width=0.55)
    fs.bar_labels(ax, bars, '{:,.0f}')
    ax.set_title(title)
    ax.set_ylabel(column)
    ax.tick_params(axis='x', length=0)
    ax.margins(y=0.12)
fig.tight_layout()
fs.save(fig, out, '1_cells_per_brain')

## 2-3. Coronal maps of cell density
Cells counted in 100 × 100 µm squares (dorsal–ventral × left–right). Same colour scale for all brains, so they can be compared. Gray line = outline of the brain.

In [ ]:
bin_um = 100

def coronal_density(d, y_range_um=None):
    """Cells per mm³ in 100 x 100 µm squares of a coronal slab (y_range_um = (front, back); None = whole brain)."""
    size_um = np.array(d['mask'].shape) * d['mask_um']
    um = d['um']
    if y_range_um is None:
        y_range_um = (0, size_um[1])
    um = um[um.y.between(*y_range_um)]
    edges_z = np.arange(0, size_um[0] + bin_um, bin_um)
    edges_x = np.arange(0, size_um[2] + bin_um, bin_um)
    counts, _, _ = np.histogram2d(um.z, um.x, bins=[edges_z, edges_x])
    thickness_mm = (y_range_um[1] - y_range_um[0]) / 1000
    extent_mm = [0, edges_x[-1] / 1000, edges_z[-1] / 1000, 0]
    return counts / ((bin_um / 1000) ** 2 * thickness_mm), extent_mm

def coronal_outline(ax, d, y_range_um=None):
    """Gray contour of the brain in a coronal slab."""
    mask = d['mask']
    if y_range_um is not None:
        i0, i1 = (int(v / d['mask_um'][1]) for v in y_range_um)
        mask = mask[:, i0:max(i1, i0 + 1)]
    projected = mask.max(axis=1)
    z_mm = (np.arange(projected.shape[0]) + 0.5) * d['mask_um'][0] / 1000
    x_mm = (np.arange(projected.shape[1]) + 0.5) * d['mask_um'][2] / 1000
    ax.contour(x_mm, z_mm, projected, levels=[0.5], colors=fs.MUTED, linewidths=0.8)

def brain_y_extent_um(d):
    """Front and back of the brain (µm along y)."""
    ys = np.flatnonzero(d['mask'].any(axis=(0, 2)))
    return ys[0] * d['mask_um'][1], (ys[-1] + 1) * d['mask_um'][1]

def colour_scale(maps):
    values = np.concatenate([m[m > 0] for m in maps] or [np.array([1.0])])
    return np.percentile(values, 99)

def colorbar(fig, image, axes, label):
    bar = fig.colorbar(image, ax=np.ravel(axes).tolist(), shrink=0.7, pad=0.02)
    bar.set_label(label, color=fs.INK_2)
    bar.outline.set_visible(False)

# ---- 2. whole brain, projected from front to back ----
if brains:
    maps = {brain: coronal_density(d) for brain, d in data.items()}
    vmax = colour_scale([m for m, _ in maps.values()])
    fig, axes = plt.subplots(1, len(brains), figsize=(3.8 * len(brains) + 1.5, 3.4), squeeze=False)
    for ax, brain in zip(axes[0], brains):
        m, extent = maps[brain]
        image = ax.imshow(m, cmap=fs.BLUE_RAMP, vmin=0, vmax=vmax, extent=extent, interpolation='nearest')
        coronal_outline(ax, data[brain])
        fs.image_axes(ax)
        ax.set_title(f"{brain}  ·  {len(data[brain]['cells']):,} cells", fontsize=11)
    fs.scale_bar(axes[0, -1], 1, 1, label='1 mm', color=fs.INK_2)
    colorbar(fig, image, axes, 'cells per mm³')
    fig.suptitle('Cell density, coronal view (whole brain, front to back)', x=0.01, ha='left', fontweight='bold')
    fs.save(fig, out, '2_density_coronal')

# ---- 3. coronal slabs from front to back ----
n_slabs, slab_um = 6, 500
if brains:
    slabs = {}
    for brain, d in data.items():
        front, back = brain_y_extent_um(d)
        centres = np.linspace(front, back, n_slabs + 2)[1:-1]
        slabs[brain] = [(c - slab_um / 2, c + slab_um / 2, (c - front) / 1000) for c in centres]
    maps = {(b, i): coronal_density(data[b], s[:2]) for b in brains for i, s in enumerate(slabs[b])}
    vmax = colour_scale([m for m, _ in maps.values()])
    fig, axes = plt.subplots(len(brains), n_slabs, figsize=(2.6 * n_slabs + 1.2, 2.0 * len(brains) + 0.8),
                             squeeze=False)
    for row, brain in enumerate(brains):
        for col, (y0, y1, from_front) in enumerate(slabs[brain]):
            ax = axes[row, col]
            m, extent = maps[(brain, col)]
            image = ax.imshow(m, cmap=fs.BLUE_RAMP, vmin=0, vmax=vmax, extent=extent, interpolation='nearest')
            coronal_outline(ax, data[brain], (y0, y1))
            fs.image_axes(ax)
            ax.set_title(f'{from_front:.1f} mm from front', fontsize=9, fontweight='normal', color=fs.INK_2)
        axes[row, 0].set_ylabel(brain, color=fs.INK, fontsize=12, fontweight='bold')
    fs.scale_bar(axes[-1, -1], 1, 1, label='1 mm', color=fs.INK_2)
    colorbar(fig, image, axes, 'cells per mm³')
    fig.suptitle(f'Cell density in coronal slabs of {slab_um} µm, front to back', x=0.01, ha='left', fontweight='bold')
    fs.save(fig, out, '3_density_coronal_slabs')

## 4. Cell density along the anterior–posterior axis
Cells per mm³ of tissue in coronal slabs of 200 µm, from the front of each brain.

In [ ]:
step_um = 200
fig, ax = plt.subplots(figsize=(9, 4.5))
for brain in brains:
    d = data[brain]
    front, back = brain_y_extent_um(d)
    rows_um = (np.arange(d['mask'].shape[1]) + 0.5) * d['mask_um'][1]
    volume_per_row = d['mask'].sum(axis=(0, 2)) * np.prod(d['mask_um']) / 1e9
    edges = np.arange(front, back + step_um, step_um)
    counts, _ = np.histogram(d['um'].y, edges)
    volume, _ = np.histogram(rows_um, edges, weights=volume_per_row)
    density = np.where(volume > 0.05 * volume.max(), counts / np.maximum(volume, 1e-9), np.nan)
    from_front_mm = ((edges[:-1] + edges[1:]) / 2 - front) / 1000
    ax.plot(from_front_mm, density, color=fs.brain_color(brain, brains), label=brain)
    last = np.flatnonzero(~np.isnan(density))
    if len(last):
        ax.annotate(brain, (from_front_mm[last[-1]], density[last[-1]]), xytext=(5, 0),
                    textcoords='offset points', va='center', color=fs.INK_2, fontsize=10)
ax.set_xlabel('anterior → posterior (mm from the front of the brain)')
ax.set_ylabel('cells per mm³')
ax.set_title('Cell density from front to back')
if len(brains) > 1:
    ax.legend(loc='upper left')
fs.save(fig, out, '4_density_anterior_posterior')

## 5. Coronal close-up: cFos image with the detected cells
By default: the place with the most cells (in the coronal view) of the first brain. Change `example_brain` or `centre` (z, y, x in full-resolution voxels) to show another place.

In [ ]:
example_brain = brains[0] if brains else None
width_um = 900          # size of the close-up (left-right and up-down)
half_slab_um = 4        # thickness of the coronal slab shown: centre ± this
cell_reach_um = 8       # cells whose centre is within ± this of the slab are drawn (~ radius of a nucleus)

d = data[example_brain]
cells, vox = d['cells'], d['voxel_um']
m, _ = coronal_density(d)
iz, ix = np.unravel_index(np.argmax(m), m.shape)
in_square = d['um'].z.between(iz * bin_um, (iz + 1) * bin_um) & d['um'].x.between(ix * bin_um, (ix + 1) * bin_um)
centre = (int((iz + 0.5) * bin_um / vox[0]), int(cells.y[in_square].median()), int((ix + 0.5) * bin_um / vox[2]))

z, y, x = centre
half_z, half_x, half_y = int(width_um / 2 / vox[0]), int(width_um / 2 / vox[2]), max(1, int(half_slab_um / vox[1]))
z0, x0 = max(0, z - half_z), max(0, x - half_x)
signal = cf.get_channel(cf.open_brain(example_brain, settings), settings['signal_channel'])
image = np.asarray(signal[z0:z0 + 2 * half_z, y - half_y:y + half_y + 1, x0:x0 + 2 * half_x]).max(1)
near = cells[(abs(cells.y - y) * vox[1] <= cell_reach_um) & cells.z.between(z0, z0 + 2 * half_z - 1)
             & cells.x.between(x0, x0 + 2 * half_x - 1)]
extent_um = [0, image.shape[1] * vox[2], image.shape[0] * vox[0], 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 6.6))
for ax, show_cells in zip(axes, [False, True]):
    ax.imshow(image, cmap='gray', vmin=np.percentile(image, 1), vmax=np.percentile(image, 99.8), extent=extent_um)
    if show_cells:
        ax.scatter((near.x - x0) * vox[2], (near.z - z0) * vox[0], s=70, facecolors='none',
                   edgecolors=fs.CELL, linewidths=1.2)
    fs.image_axes(ax)
    fs.scale_bar(ax, 100, 1)
axes[0].set_title('cFos')
axes[1].set_title(f'detected cells ({len(near)})')
fs.title(fig, f'{example_brain}, coronal close-up of {width_um / 1000:.1f} mm '
              f'({(y * vox[1] - brain_y_extent_um(d)[0]) / 1000:.1f} mm from the front, dorsal up)')
fs.save(fig, out, '5_close_up_coronal')

## 6. Video: flying from front to back through the close-up
The same coronal close-up, slice by slice from anterior to posterior (about 10 s). Each detected cell is drawn as a ring that grows and shrinks as we pass through it, like a sphere. Saved as `6_fly_through_coronal.mp4` (plays in PowerPoint).

In [ ]:
import imageio.v2 as imageio

n_frames = 150         # coronal slices, one per frame (about 10 s at 15 frames per second)
fps = 15
reach = 4              # a cell is drawn on the slices within ±4 pixels (~6 µm) of its centre

first = max(0, y - n_frames // 2)
volume = np.asarray(signal[z0:z0 + 2 * half_z, first:first + n_frames, x0:x0 + 2 * half_x])
vmin, vmax = np.percentile(volume, 1), np.percentile(volume, 99.8)
in_box = cells[cells.y.between(first - reach, first + n_frames + reach)
               & cells.z.between(z0, z0 + 2 * half_z - 1) & cells.x.between(x0, x0 + 2 * half_x - 1)]

fig, ax = plt.subplots(figsize=(8, 8), dpi=100)   # 800 x 800 pixels
fig.subplots_adjust(0, 0, 1, 1)
writer = imageio.get_writer(out / '6_fly_through_coronal.mp4', fps=fps, codec='libx264', quality=8)
for i in range(volume.shape[1]):
    ax.clear()
    ax.imshow(volume[:, i, :], cmap='gray', vmin=vmin, vmax=vmax, extent=extent_um)
    distance = np.abs(in_box.y.to_numpy() - (first + i))
    visible = distance <= reach
    shrink = 1 - (distance[visible] / (reach + 1)) ** 2          # like the section of a sphere
    ax.scatter((in_box.x[visible] - x0) * vox[2], (in_box.z[visible] - z0) * vox[0], s=110 * shrink,
               facecolors='none', edgecolors=fs.CELL, linewidths=1.3)
    fs.image_axes(ax)
    fs.scale_bar(ax, 100, 1)
    ax.text(0.03, 0.97, f'{example_brain}   coronal, {i * vox[1]:.0f} µm', transform=ax.transAxes,
            color='white', fontsize=14, va='top')
    fig.canvas.draw()
    writer.append_data(np.asarray(fig.canvas.buffer_rgba())[..., :3])
writer.close()
plt.close(fig)
print('saved', out / '6_fly_through_coronal.mp4')

## 7. 3D view: raw cFos, cell density and cells
The brain is resampled to cubes of `grid_um` and seen from the front-side and slightly from above (dorsal up). Gray = raw cFos (brightest value along each line of sight), blue glow = cell density, orange dots = a random sample of the detected cells. Rendered without a graphics card (`brain_3d.py`), so it also runs in the background.

In [ ]:
from cfos_mapper.figures import brain_3d as b3
from matplotlib.patches import Polygon

brain_3d_name = example_brain     # brain shown in 3D (sections 7-9)
grid_um = 30                      # size of the 3D cubes for the still and the fly-through (smaller = sharper, slower)
n_cells_shown = 20000             # random sample of cells drawn as dots in 3D
view_angles = dict(azimuth=35, elevation=20)

d3 = data[brain_3d_name]
dataset = cf.open_brain(brain_3d_name, settings)
level = 3                         # low-resolution copy (24 x 11.7 x 11.7 µm) is enough for 30 µm cubes
low_res = cf.load_into_memory(cf.get_channel(dataset, settings['signal_channel'], level))

def make_3d(um):
    """cFos and cell density on cubes of `um`, with the cFos contrast range."""
    cfos = b3.isotropic(low_res, cf.voxel_size(dataset, level), um)
    density = b3.density_volume(d3['um'], cfos.shape, um)
    return dict(um=um, cfos=cfos, density=density,
                cfos_range=(np.percentile(cfos, 60), np.percentile(cfos, 99.95)))

def label_3d(ax, projector, shape, colour='white'):
    """Names of the directions, just outside the brain."""
    Z, Y, X = shape
    for text, point in [('anterior', (Z / 2, -0.06 * Y, X / 2)), ('posterior', (Z / 2, 1.06 * Y, X / 2)),
                        ('dorsal', (-0.12 * Z, Y / 2, X / 2))]:
        rows, cols = projector.points(np.array([point]))
        ax.text(cols[0], rows[0], text, color=colour, fontsize=11, ha='center', va='center', alpha=0.85)

def crop_to_brain(ax, image, margin=15):
    rows, cols = np.nonzero(image.sum(axis=2) > 0.05)
    ax.set_xlim(cols.min() - margin, cols.max() + margin)
    ax.set_ylim(rows.max() + margin, rows.min() - margin)

rng = np.random.default_rng(0)
sample = d3['um'].iloc[rng.choice(len(d3['um']), min(n_cells_shown, len(d3['um'])), replace=False)]

fine = make_3d(grid_um)
projector = b3.Projector(fine['cfos'].shape, b3.view(**view_angles))
density_image = projector.volume(fine['density'])
fine['density_limits'] = b3.density_range(density_image)   # only the densest parts glow
base_3d = b3.composite(projector.volume(fine['cfos']), density_image, fine['cfos_range'], fine['density_limits'])

fig, ax = plt.subplots(figsize=(10, 8), facecolor='black')
ax.set_facecolor('black')
ax.imshow(base_3d, interpolation='bilinear')
rows, cols = projector.points(sample[['z', 'y', 'x']].to_numpy() / grid_um)
ax.scatter(cols, rows, s=0.6, color=fs.CELL, alpha=0.6, linewidths=0)
label_3d(ax, projector, fine['cfos'].shape)
crop_to_brain(ax, base_3d)
fs.image_axes(ax)
ax.set_title(f'{brain_3d_name}: raw cFos (gray), cell density (blue), cells (orange, {len(sample):,} of '
             f'{len(d3["um"]):,})', color='white', loc='left', fontsize=12)
fs.save(fig, out, '7_3D_view')

## 8. Video: front-to-back fly-through, 2D section next to the 3D brain
Left: the coronal section of raw cFos (dorsal up) with the cells detected in it (orange). Right: the 3D brain; the light-blue plane shows where the section is, and the cells near it light up in orange. Saved as `8_fly_through_2D_3D.mp4` (1600 × 720, plays in PowerPoint).

In [ ]:
step_um = 50            # distance between two frames, front -> back
fps = 15
half_slab_um = 8        # 2D: cells within ± this of the section are drawn (~ radius of a nucleus)
half_slab_3d_um = 25    # 3D: cells within ± this of the plane light up

# coronal sections from the medium-resolution copy (12 x 5.85 x 5.85 µm), kept in memory (~4 GB)
level_2d = 2
section_um = np.array(cf.voxel_size(dataset, level_2d))
sections = cf.load_into_memory(cf.get_channel(dataset, settings['signal_channel'], level_2d))
sample_values = sections[::4, ::8, ::8]
section_range = (np.percentile(sample_values, 50), np.percentile(sample_values, 99.9))
section_extent_mm = [0, sections.shape[2] * section_um[2] / 1000, sections.shape[0] * section_um[0] / 1000, 0]

# cells sorted front -> back, to find the ones near each section quickly
cells_um = d3['um'].sort_values('y')
cells_y = cells_um.y.to_numpy()
cells_zx_mm = cells_um[['x', 'z']].to_numpy() / 1000
cells_grid = cells_um[['z', 'y', 'x']].to_numpy() / grid_um

def near(y_um, half):
    i0, i1 = np.searchsorted(cells_y, [y_um - half, y_um + half])
    return slice(i0, i1)

front, back = brain_y_extent_um(d3)
Z, Y, X = fine['cfos'].shape

fig = plt.figure(figsize=(16, 7.2), dpi=100, facecolor='black')   # 1600 x 720 pixels
ax2d = fig.add_axes([0.01, 0.05, 0.56, 0.86])
ax3d = fig.add_axes([0.58, 0.02, 0.41, 0.94])
for ax in (ax2d, ax3d):
    fs.image_axes(ax)
    ax.set_facecolor('black')
section_image = ax2d.imshow(sections[:, 0, :], cmap='gray', vmin=section_range[0], vmax=section_range[1],
                            extent=section_extent_mm)
dots_2d = ax2d.scatter([], [], s=4, color=fs.CELL, linewidths=0)
fs.scale_bar(ax2d, 1, 1, label='1 mm')
title_2d = ax2d.set_title('', color='white', loc='left', fontsize=13)
ax3d.imshow(base_3d, interpolation='bilinear')
plane = Polygon(np.zeros((4, 2)), closed=True, facecolor=(0.53, 0.71, 0.94, 0.18), edgecolor='#86b6ef', linewidth=1.2)
ax3d.add_patch(plane)
dots_3d = ax3d.scatter([], [], s=1.5, color=fs.CELL, linewidths=0)
label_3d(ax3d, projector, fine['cfos'].shape)
crop_to_brain(ax3d, base_3d)
fig.text(0.01, 0.015, 'orange = detected cFos+ cells   ·   blue glow = cell density', color='#c3c2b7', fontsize=11)

writer = imageio.get_writer(out / '8_fly_through_2D_3D.mp4', fps=fps, codec='libx264', quality=8)
for y_um in np.arange(front, back, step_um):
    section_image.set_data(sections[:, min(int(y_um / section_um[1]), sections.shape[1] - 1), :])
    dots_2d.set_offsets(cells_zx_mm[near(y_um, half_slab_um)])
    title_2d.set_text(f'{brain_3d_name}   coronal section, {(y_um - front) / 1000:.2f} mm from the front')
    corners = np.array([[0, y_um / grid_um, 0], [0, y_um / grid_um, X - 1],
                        [Z - 1, y_um / grid_um, X - 1], [Z - 1, y_um / grid_um, 0]])
    rows, cols = projector.points(corners)
    plane.set_xy(np.c_[cols, rows])
    rows, cols = projector.points(cells_grid[near(y_um, half_slab_3d_um)])
    dots_3d.set_offsets(np.c_[cols, rows])
    fig.canvas.draw()
    writer.append_data(np.asarray(fig.canvas.buffer_rgba())[..., :3])
writer.close()
plt.close(fig)
print('saved', out / '8_fly_through_2D_3D.mp4')

## 9. Video: the 3D brain turning around
One full turn around the vertical axis (dorsal stays up). Uses coarser cubes (`turn_um`) so that it takes ~20-30 minutes instead of hours. Saved as `9_turning_3D.mp4` (800 × 800).

In [ ]:
turn_um = 50            # cube size for the turning video (bigger = faster, less sharp)
n_frames = 120          # 3 degrees per frame: 8 s at 15 frames per second
elevation = 20

coarse = make_3d(turn_um)
sample_grid = sample[['z', 'y', 'x']].to_numpy() / turn_um
# one colour range for the density in all frames (no flicker): from the first view
coarse['density_limits'] = b3.density_range(
    b3.Projector(coarse['density'].shape, b3.view(azimuth=0, elevation=elevation)).volume(coarse['density']))

fig, ax = plt.subplots(figsize=(8, 8), dpi=100, facecolor='black')   # 800 x 800 pixels
fig.subplots_adjust(0, 0, 1, 1)
writer = imageio.get_writer(out / '9_turning_3D.mp4', fps=fps, codec='libx264', quality=8)
for azimuth in np.linspace(0, 360, n_frames, endpoint=False):
    turning = b3.Projector(coarse['cfos'].shape, b3.view(azimuth=azimuth, elevation=elevation))
    image = b3.composite(turning.volume(coarse['cfos']), turning.volume(coarse['density']),
                         coarse['cfos_range'], coarse['density_limits'])
    ax.clear()
    ax.set_facecolor('black')
    ax.imshow(image, interpolation='bilinear')
    rows, cols = turning.points(sample_grid)
    ax.scatter(cols, rows, s=0.5, color=fs.CELL, alpha=0.6, linewidths=0)
    label_3d(ax, turning, coarse['cfos'].shape)
    fs.image_axes(ax)
    ax.text(0.03, 0.97, f'{brain_3d_name}: raw cFos, cell density (blue), cells (orange)', transform=ax.transAxes,
            color='white', fontsize=12, va='top')
    fig.canvas.draw()
    writer.append_data(np.asarray(fig.canvas.buffer_rgba())[..., :3])
writer.close()
plt.close(fig)
print('saved', out / '9_turning_3D.mp4')